In [0]:
df = spark.read.format("csv").option("header",True).load("/Volumes/practice/default/data_sample/titanic_data.csv")

In [0]:
df.display()

#1 Calculate Running total

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
ex1 = (
    df.groupBy("embark_town").agg(
        sum(col("survived").cast('int')).alias("Survived")
    )
    .orderBy("Survived")
)
ex1.display()

In [0]:
## Window

window_spec = Window.orderBy("embark_town").rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)

run_total = ex1.withColumn(
    "Running_tot",
    sum("Survived").over(window_spec)
)
run_total.display()

In [0]:
df.createOrReplaceTempView("sql_table")

In [0]:
%sql
-- Same in SQL

with cte1 as (
select 
embark_town,
sum(cast(survived as int)) as Survived
from sql_table
group by 1
order by 1)
select
*,
sum(survived) over(order by embark_town) as Running_tot
from cte1